# Flybrain — Kaggle / Colab GPU runner (minimal)

Paste these cells into a **new** Kaggle notebook (old saved copies ignore repo edits).

1. Attach Input → dataset with the modules (prefer `kaggle_data_v2` / updated `brain-data`).
2. Accelerator → GPU T4.
3. Run all.

Repo: https://github.com/Cyberisthename/brain-experiments


In [ ]:
# --- 1) Clone / reset + deps + symlink ---
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Cyberisthename/brain-experiments"
BASE = Path("/kaggle/working") if Path("/kaggle/working").exists() else (
    Path("/content") if Path("/content").exists() else Path.cwd()
)
REPO = BASE / "brain-experiments"

if not (REPO / "gpu_core.py").exists():
    !git clone --depth 1 {REPO_URL} {REPO}
else:
    !git -C {REPO} fetch --depth 1 origin main
    !git -C {REPO} reset --hard origin/main

%cd {REPO}
!pip -q install cupy-cuda12x scipy numpy pandas pyarrow scikit-learn huggingface_hub

Path("/workspace").mkdir(parents=True, exist_ok=True)
link = Path("/workspace/flybrain")
if not link.exists():
    link.symlink_to(REPO)

sys.path[:0] = [str(REPO), str(REPO / "compress")]
os.environ["FLYBRAIN_GPU"] = "1"
sha = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("HEAD", sha)
print("cwd", Path.cwd())


In [ ]:
# --- 2) Data setup (one liner) ---
# Copies matching files from /kaggle/input/** into expected repo paths; prints OK/MISS
!python scripts/kaggle_setup.py


In [ ]:
# --- 3) GPU self-check ---
import gpu_core as GC
print("cupy_available:", GC.cupy_available())
if not GC.cupy_available():
    raise SystemExit("No GPU/CuPy — set Accelerator → GPU T4, then Run all again.")
import cupy as cp
print("device:", cp.cuda.Device(0), "cupy:", cp.__version__)
GC.enable()
print("using_gpu:", GC.using_gpu())
!python3 verify_gpu_core.py


In [ ]:
# --- 4) Run experiment ---
import os, time, json
from pathlib import Path
from datetime import datetime, timezone
import gpu_core as GC
GC.enable()

EXPERIMENT = "myelin_real_rep"  # or mega_franken / trained_bridges
# Full prereg run by default. For a short smoke on Kaggle first:
# os.environ["MYELIN_SMOKE"] = "1"

runners = {
    "myelin_real_rep": "regions/myelin_real_rep/myelin_real_rep.py",
    "mega_franken": "regions/mega_franken/mega_franken.py",
    "trained_bridges": "regions/trained_bridges/trained_bridges.py",
}
script = Path(runners[EXPERIMENT])
assert script.exists(), script
t0 = time.time()
print("start", EXPERIMENT, datetime.now(timezone.utc).isoformat())
!python3 -u {script}
wall = time.time() - t0
Path("out_gpu_core").mkdir(exist_ok=True)
Path("out_gpu_core/last_run.json").write_text(json.dumps({
    "experiment": EXPERIMENT, "wall_sec": wall,
    "utc": datetime.now(timezone.utc).isoformat(),
}, indent=2))
print(f"wall_sec={wall:.1f}")


In [ ]:
# --- 5) Zip results (no files.download — use Kaggle Output browser) ---
from pathlib import Path
import json, zipfile

EXPERIMENT = json.loads(Path("out_gpu_core/last_run.json").read_text())["experiment"]
out_dirs = {
    "myelin_real_rep": Path("regions/myelin_real_rep/out"),
    "mega_franken": Path("regions/mega_franken/out"),
    "trained_bridges": Path("regions/trained_bridges/out"),
}
bundle = Path("results_bundle.zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for extra in ("out_gpu_core/verify_report.json", "out_gpu_core/last_run.json"):
        p = Path(extra)
        if p.exists():
            z.write(p, p.name)
    od = out_dirs[EXPERIMENT]
    if od.exists():
        for pth in od.rglob("*"):
            if pth.is_file() and pth.stat().st_size < 50_000_000:
                z.write(pth, f"{EXPERIMENT}/{pth.relative_to(od)}")
print("wrote", bundle.resolve(), "mb=", round(bundle.stat().st_size / 1e6, 3))
print("Kaggle: Output → results_bundle.zip")
